In [1]:
from pathlib import Path
import pandas as pd
import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_CUTOFF = pd.Timestamp("2016-11-30")
EXPIRY_START = pd.Timestamp("2016-12-01")
EXPIRY_END = pd.Timestamp("2016-12-31")
OUTCOME_END = EXPIRY_END + pd.Timedelta(days=30)

con = duckdb.connect()

print("Training feature cutoff:", TRAIN_CUTOFF.date())
print("Candidate expiry range:", EXPIRY_START.date(), EXPIRY_END.date())
print("Outcome observation end:", OUTCOME_END.date())

Training feature cutoff: 2016-11-30
Candidate expiry range: 2016-12-01 2016-12-31
Outcome observation end: 2017-01-30


In [2]:
candidate_sql = """
WITH history AS (
    SELECT *
    FROM read_csv(?, all_varchar = true)
    WHERE TRY_STRPTIME(transaction_date, '%Y%m%d')::DATE
          <= CAST(? AS DATE)
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY msno
            ORDER BY
                transaction_date DESC,
                (
                    plan_list_price
                    || payment_plan_days
                    || payment_method_id
                ) ASC,
                is_cancel DESC,
                CASE
                    WHEN is_cancel = '1'
                    THEN membership_expire_date
                END ASC,
                CASE
                    WHEN is_cancel = '0'
                    THEN membership_expire_date
                END DESC
        ) AS row_rank
    FROM history
)
SELECT
    msno,
    TRY_STRPTIME(
        transaction_date, '%Y%m%d'
    )::DATE AS last_transaction_day,
    TRY_STRPTIME(
        membership_expire_date, '%Y%m%d'
    )::DATE AS candidate_expiration
FROM ranked
WHERE row_rank = 1
  AND TRY_STRPTIME(membership_expire_date, '%Y%m%d')::DATE
      BETWEEN CAST(? AS DATE) AND CAST(? AS DATE)
"""

historical_candidates = con.execute(
    candidate_sql,
    [
        str(RAW_DIR / "transactions.csv"),
        TRAIN_CUTOFF.date(),
        EXPIRY_START.date(),
        EXPIRY_END.date(),
    ],
).df()

print("Historical candidate users:", len(historical_candidates))

Historical candidate users: 882127


In [3]:
assert historical_candidates["msno"].is_unique

assert historical_candidates["last_transaction_day"].le(
    TRAIN_CUTOFF
).all()

assert historical_candidates["candidate_expiration"].between(
    EXPIRY_START,
    EXPIRY_END,
).all()

candidate_summary = pd.Series({
    "candidate_users": len(historical_candidates),
    "earliest_expiration": (
        historical_candidates["candidate_expiration"].min()
    ),
    "latest_expiration": (
        historical_candidates["candidate_expiration"].max()
    ),
    "latest_history_transaction": (
        historical_candidates["last_transaction_day"].max()
    ),
})

candidate_summary

candidate_users                            882127
earliest_expiration           2016-12-01 00:00:00
latest_expiration             2016-12-31 00:00:00
latest_history_transaction    2016-11-30 00:00:00
dtype: object

In [4]:
historical_label_sql = """
WITH ordered AS (
    SELECT
        t.*,
        TRY_STRPTIME(
            t.transaction_date, '%Y%m%d'
        )::DATE AS transaction_day,
        TRY_STRPTIME(
            t.membership_expire_date, '%Y%m%d'
        )::DATE AS expiration_day,
        ROW_NUMBER() OVER (
            PARTITION BY t.msno
            ORDER BY
                t.transaction_date ASC,
                (
                    t.plan_list_price
                    || t.payment_plan_days
                    || t.payment_method_id
                ) DESC,
                t.is_cancel ASC,
                CASE
                    WHEN t.is_cancel = '1'
                    THEN t.membership_expire_date
                END DESC,
                CASE
                    WHEN t.is_cancel = '0'
                    THEN t.membership_expire_date
                END ASC
        ) AS event_order
    FROM future_transactions AS t
    INNER JOIN label_candidates AS c
        ON t.msno = c.msno
    WHERE TRY_STRPTIME(t.transaction_date, '%Y%m%d')::DATE
          > CAST(? AS DATE)
      AND TRY_STRPTIME(t.transaction_date, '%Y%m%d')::DATE
          <= CAST(? AS DATE)
),
first_renewal AS (
    SELECT
        msno,
        transaction_day AS renewal_day,
        event_order
    FROM ordered
    WHERE is_cancel = '0'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY msno ORDER BY event_order
    ) = 1
),
adjusted AS (
    SELECT
        c.msno,
        c.candidate_expiration,
        r.renewal_day,
        LEAST(
            c.candidate_expiration,
            COALESCE(
                MIN(o.expiration_day) FILTER (
                    WHERE o.is_cancel = '1'
                      AND (
                          r.event_order IS NULL
                          OR o.event_order < r.event_order
                      )
                ),
                c.candidate_expiration
            )
        ) AS adjusted_expiration
    FROM label_candidates AS c
    LEFT JOIN first_renewal AS r
        ON c.msno = r.msno
    LEFT JOIN ordered AS o
        ON c.msno = o.msno
    GROUP BY
        c.msno,
        c.candidate_expiration,
        r.renewal_day,
        r.event_order
),
gaps AS (
    SELECT
        *,
        DATE_DIFF(
            'day',
            adjusted_expiration,
            renewal_day
        ) AS renewal_gap_days
    FROM adjusted
)
SELECT
    *,
    CASE
        WHEN renewal_day IS NULL THEN 1
        WHEN renewal_gap_days >= 30 THEN 1
        ELSE 0
    END AS is_churn
FROM gaps
"""

In [5]:
toy_candidates = pd.DataFrame({
    "msno": [
        "no_renewal",
        "gap_29",
        "gap_30",
        "cancel_then_renew",
        "early_renewal",
        "same_day_actions",
    ],
    "candidate_expiration": pd.to_datetime([
        "2016-12-01",
        "2016-12-01",
        "2016-12-01",
        "2016-12-20",
        "2016-12-20",
        "2016-12-20",
    ]),
})

toy_events = [
    ("gap_29", "20161230", "20170129", "0"),
    ("gap_30", "20161231", "20170130", "0"),
    ("cancel_then_renew", "20161205", "20161205", "1"),
    ("cancel_then_renew", "20170104", "20170203", "0"),
    ("early_renewal", "20161210", "20170109", "0"),
    ("same_day_actions", "20161205", "20161205", "1"),
    ("same_day_actions", "20161205", "20170104", "0"),
]

toy_transactions = pd.DataFrame(
    toy_events,
    columns=[
        "msno",
        "transaction_date",
        "membership_expire_date",
        "is_cancel",
    ],
)

for column, value in {
    "payment_method_id": "41",
    "payment_plan_days": "30",
    "plan_list_price": "149",
}.items():
    toy_transactions[column] = value

con.register("label_candidates", toy_candidates)
con.register("future_transactions", toy_transactions)

toy_labels = con.execute(
    historical_label_sql,
    [TRAIN_CUTOFF.date(), OUTCOME_END.date()],
).df()

expected_labels = {
    "no_renewal": 1,
    "gap_29": 0,
    "gap_30": 1,
    "cancel_then_renew": 1,
    "early_renewal": 0,
    "same_day_actions": 0,
}

actual_labels = toy_labels.set_index("msno")["is_churn"].to_dict()

assert actual_labels == expected_labels, (
    f"Unexpected labels: {actual_labels}"
)

print("All six label checks passed.")

toy_labels[
    [
        "msno",
        "adjusted_expiration",
        "renewal_day",
        "renewal_gap_days",
        "is_churn",
    ]
].sort_values("msno")

All six label checks passed.


,msno,adjusted_expiration,renewal_day,renewal_gap_days,is_churn
4,cancel_then_renew,2016-12-05,2017-01-04,30,1
1,early_renewal,2016-12-20,2016-12-10,-10,0
3,gap_29,2016-12-01,2016-12-30,29,0
5,gap_30,2016-12-01,2016-12-31,30,1
0,no_renewal,2016-12-01,NaT,<NA>,1
2,same_day_actions,2016-12-20,2016-12-05,-15,0


In [6]:
# Replace the toy tables with the actual candidates and transactions.
con.unregister("label_candidates")
con.unregister("future_transactions")

con.register("label_candidates", historical_candidates)

con.read_csv(
    str(RAW_DIR / "transactions.csv"),
    all_varchar=True,
).create_view("future_transactions")

historical_label_details = con.execute(
    historical_label_sql,
    [TRAIN_CUTOFF.date(), OUTCOME_END.date()],
).df()

historical_labels = historical_label_details[
    ["msno", "is_churn"]
].copy()

print("Historical labeled users:", len(historical_labels))

Historical labeled users: 882127


In [7]:
assert len(historical_labels) == len(historical_candidates)
assert historical_labels["msno"].is_unique
assert historical_labels["is_churn"].notna().all()
assert historical_labels["is_churn"].isin([0, 1]).all()

assert (
    historical_label_details["candidate_expiration"]
    + pd.Timedelta(days=30)
).le(OUTCOME_END).all()

renewal_days = historical_label_details["renewal_day"].dropna()

assert renewal_days.gt(TRAIN_CUTOFF).all()
assert renewal_days.le(OUTCOME_END).all()

VALIDATION_CUTOFF = pd.Timestamp("2017-02-28")
assert OUTCOME_END < VALIDATION_CUTOFF

historical_label_summary = pd.Series({
    "users": len(historical_labels),
    "churned_users": int(historical_labels["is_churn"].sum()),
    "churn_rate": float(historical_labels["is_churn"].mean()),
    "users_without_observed_renewal": int(
        historical_label_details["renewal_day"].isna().sum()
    ),
    "users_with_negative_renewal_gap": int(
        historical_label_details["renewal_gap_days"].lt(0).sum()
    ),
    "latest_observed_renewal": renewal_days.max(),
})

historical_label_summary

users                                           882127
churned_users                                    85237
churn_rate                                    0.096627
users_without_observed_renewal                   66442
users_with_negative_renewal_gap                  91682
latest_observed_renewal            2017-01-30 00:00:00
dtype: object

In [8]:
import json

processed_dir = PROJECT_ROOT / "data" / "processed"
report_dir = PROJECT_ROOT / "reports"
sql_dir = PROJECT_ROOT / "sql"

for directory in [processed_dir, report_dir, sql_dir]:
    directory.mkdir(parents=True, exist_ok=True)

con.from_df(historical_labels).write_parquet(
    str(processed_dir / "historical_labels_dec2016.parquet")
)

con.from_df(historical_label_details).write_parquet(
    str(processed_dir / "historical_label_diagnostics_dec2016.parquet")
)

(sql_dir / "historical_candidates.sql").write_text(
    candidate_sql.strip() + "\n",
    encoding="utf-8",
)

(sql_dir / "historical_labels.sql").write_text(
    historical_label_sql.strip() + "\n",
    encoding="utf-8",
)

historical_label_summary.rename("value").to_csv(
    report_dir / "historical_label_summary.csv",
    index_label="metric",
)

label_config = {
    "label_source": "reconstructed",
    "transaction_source": "transactions.csv",
    "history_cutoff": str(TRAIN_CUTOFF.date()),
    "expiry_start": str(EXPIRY_START.date()),
    "expiry_end": str(EXPIRY_END.date()),
    "outcome_observation_end": str(OUTCOME_END.date()),
    "churn_gap_threshold_days": 30,
    "churn_rule": "no observed renewal or renewal gap >= 30 days",
    "reference": "WSDMChurnLabeller.scala",
    "toy_cases_passed": 6,
    "official_label_parity_verified": False,
}

(report_dir / "historical_label_config.json").write_text(
    json.dumps(label_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Historical labels, diagnostics, SQL, and configuration saved.")

Historical labels, diagnostics, SQL, and configuration saved.


## Historical Label Summary

Reconstructed labels for 882,127 users whose membership expiration dates, determined from history through November 30, 2016, fall in December 2016.

Outcome transactions are observed through January 30, 2017. The reconstructed churn rate is 9.66%: 66,442 users have no observed renewal, and 18,795 have a renewal gap of at least 30 days.

Six synthetic checks passed. Exact parity with the original Scala implementation and official labels has not been independently verified. Future-derived diagnostic fields are excluded from model inputs.